# Task 01: Data Preparation and Baseline

**Project:** Telco Customer Churn Prediction  
**Objective:** Prepare a public dataset, create a simple baseline, and report a reliable test accuracy score.

This notebook follows the task requirements:
1. Separate the target and features.
2. Split the data into training and testing sets **before preprocessing or exploratory analysis**.
3. Build a trivial baseline using `DummyClassifier`.
4. Train a simple model that can be compared with the baseline.
5. Report accuracy on the untouched test set.

**Dataset:** Telco Customer Churn dataset. The expected target column is `Churn Label` with values `Yes` and `No`.


## 1. Setup

Install the required libraries if they are not already available. The notebook uses pandas and scikit-learn only.

In [ ]:
# If needed, uncomment this line in Google Colab:
# %pip install -q pandas scikit-learn matplotlib seaborn

import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries imported successfully.")

## 2. Load the public dataset

Place the CSV file in the same folder as this notebook. The code checks common file names. In Google Colab, it will ask you to upload the CSV if it cannot find it.

In [ ]:
candidate_paths = [
    "telco.csv",
    "telco[1].csv",
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "/content/telco.csv",
    "/content/telco[1].csv"
]

DATA_PATH = next((path for path in candidate_paths if os.path.exists(path)), None)

if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("Please upload a CSV dataset.")
        DATA_PATH = csv_files[0]
    except ImportError:
        raise FileNotFoundError(
            "CSV not found. Place the dataset CSV beside this notebook "
            "and update DATA_PATH if its filename is different."
        )

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Dataset shape: {df.shape}")
display(df.head())

## 3. Define features and target, then split immediately

The target is `Churn Label`. The split happens before checking missing values, encoding categories, scaling values, or doing exploratory analysis.

Some columns are excluded because they identify a customer, contain location identifiers, or describe churn outcomes/scores that could reveal the target. This helps avoid target leakage.

In [ ]:
TARGET_COLUMN = "Churn Label"

if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        f"Available columns: {df.columns.tolist()}"
    )

# Convert the target into numeric labels: No = 0, Yes = 1.
y = df[TARGET_COLUMN].astype(str).str.strip().str.lower().map({
    "no": 0,
    "yes": 1
})

if y.isna().any():
    unexpected = df.loc[y.isna(), TARGET_COLUMN].unique()
    raise ValueError(f"Unexpected target labels found: {unexpected}")

# Remove the target and columns that may cause leakage or add little value.
columns_to_drop = [
    "Customer ID",
    "Customer Status",
    "Churn Score",
    "Churn Category",
    "Churn Reason",
    "CLTV",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Latitude",
    "Longitude",
    "Population",
    "Quarter"
]

X = df.drop(columns=[TARGET_COLUMN] + columns_to_drop, errors="ignore").copy()

# IMPORTANT: Split before preprocessing or exploratory analysis.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train/test split completed.")
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).rename("proportion"))
print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).rename("proportion"))

## 4. Inspect the training data

Exploratory checks below use the training set. The test set remains untouched until final evaluation.

In [ ]:
display(X_train.head())
print("Training data information:")
X_train.info()

print("\nMissing values in training features:")
missing_counts = X_train.isna().sum().sort_values(ascending=False)
display(missing_counts[missing_counts > 0])

print("\nDuplicate rows in training features:", X_train.duplicated().sum())

In [ ]:
# Check the target balance in the training set.
target_counts = y_train.value_counts().sort_index()
target_counts.index = ["No churn (0)", "Churn (1)"]

target_counts.plot(kind="bar", figsize=(6, 4))
plt.title("Training Target Distribution")
plt.ylabel("Number of customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Build preprocessing steps

Preprocessing is placed inside a scikit-learn pipeline. This ensures that imputers, encoders, and the scaler learn from training data only. The same fitted transformations are then applied to the test data.

- Numeric columns: median imputation and standard scaling.
- Categorical columns: most-frequent imputation and one-hot encoding.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 6. Trivial baseline model

A `DummyClassifier` with the `most_frequent` strategy always predicts the most common class in the training data. It does not learn customer patterns. Its test accuracy is the minimum benchmark that a useful model should beat.

In [ ]:
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline test accuracy: {baseline_accuracy:.4f} ({baseline_accuracy:.2%})")

## 7. Train a simple model

Logistic Regression is used as a deliberately simple classification model. It is trained through the preprocessing pipeline, so all transformations are fitted only on the training data.

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)

test_predictions = model.predict(X_test)
model_accuracy = accuracy_score(y_test, test_predictions)

print(f"Logistic Regression test accuracy: {model_accuracy:.4f} ({model_accuracy:.2%})")

## 8. Compare the baseline and model

The primary result for this task is the model's accuracy on the held-out test set. Compare it with the trivial baseline to check whether the model adds value.

In [ ]:
results = pd.DataFrame({
    "Model": ["Dummy baseline", "Logistic Regression"],
    "Test accuracy": [baseline_accuracy, model_accuracy]
})

display(results)

if model_accuracy > baseline_accuracy:
    print("Result: Logistic Regression beats the trivial baseline.")
elif model_accuracy == baseline_accuracy:
    print("Result: Logistic Regression matches the trivial baseline.")
else:
    print("Result: Logistic Regression does not beat the trivial baseline yet.")

## 9. Final evaluation

The following report gives additional context about the final model. Accuracy remains the main score reported for the task. The confusion matrix shows correct and incorrect predictions for each class.

In [ ]:
print("Classification report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["No churn", "Churn"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=["No churn", "Churn"],
    cmap="Blues"
)
plt.title("Logistic Regression: Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 10. Conclusion

Fill in the final values after running all cells.

- Dataset: Telco Customer Churn.
- Target: `Churn Label` (`No = 0`, `Yes = 1`).
- Train/test split: 80% training and 20% testing, stratified by the target.
- Baseline: `DummyClassifier(strategy="most_frequent")`.
- Simple model: Logistic Regression.
- Baseline test accuracy: printed in Section 6.
- Logistic Regression test accuracy: printed in Section 7.

**Main result:** Report the Logistic Regression test accuracy as the model score, and state whether it beats the dummy baseline.

**Reproducibility:** The random state is fixed at 42. The test set is held out from preprocessing and model training.